# 06. Комбінація інженерних ознак

**Етап 06 з 13** · NASA C-MAPSS FD001 · RUL prediction


## 1. Дані

In [ ]:
%pip install -q PyWavelets

import os
from pathlib import Path

import numpy as np
import pandas as pd
import pywt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor

DATA_DIR = Path(os.environ.get("FD001_DIR", "/content/drive/MyDrive/predictive_maintenance"))

SENSORS = [f"sensor_{i}" for i in range(1, 22)]
COLUMNS = ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"] + SENSORS

train = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)

# RUL = останній цикл двигуна - поточний цикл
train["RUL"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(train, groups=train["unit_id"]))

train_df = train.iloc[train_idx].copy()
val_df = train.iloc[val_idx].copy()

y_train = train_df["RUL"]
y_val = val_df["RUL"]

print("Train engines:", train_df["unit_id"].nunique(), "| rows:", len(train_df))
print("Validation engines:", val_df["unit_id"].nunique(), "| rows:", len(val_df))

## 2. Ознаки

In [ ]:
BASELINE_FEATURES = ["cycle", "setting_1", "setting_2", "setting_3"] + SENSORS

TOP8_SENSORS = [
    "sensor_11", "sensor_4", "sensor_12", "sensor_7",
    "sensor_9", "sensor_15", "sensor_20", "sensor_21",
]

In [ ]:
WAVELET_SENSORS = ["sensor_4", "sensor_7", "sensor_11", "sensor_12", "sensor_15", "sensor_20", "sensor_21"]
WAVELET_WINDOW = 16


def add_wavelet_d2_std(df):
    """
    Додає std деталей рівня 2 (db1) у ковзному вікні з 16 точок для кожного сенсора.
    Вікно причинне: початок двигуна доповнюється першим значенням (edge).
    """

    result = df.copy()
    feature_frames = []

    for _, group in result.groupby("unit_id", sort=False):
        group = group.sort_values("cycle")
        features = {}

        for sensor in WAVELET_SENSORS:
            values = group[sensor].to_numpy(dtype=float)
            d2_std = []

            for i in range(len(values)):
                history = values[max(0, i - WAVELET_WINDOW + 1):i + 1]

                if len(history) < WAVELET_WINDOW:
                    history = np.pad(history, (WAVELET_WINDOW - len(history), 0), mode="edge")

                coeffs = pywt.wavedec(history, wavelet="db1", level=2, mode="symmetric")
                # coeffs = [cA2, cD2, cD1]
                d2_std.append(np.std(coeffs[1]))

            features[f"{sensor}_wlt_d2_std"] = d2_std

        feature_frames.append(pd.DataFrame(features, index=group.index))

    wavelet_features = pd.concat(feature_frames).sort_index()

    for col in wavelet_features.columns:
        result[col] = wavelet_features[col]

    return result

In [ ]:
class DegradationDetector:
    """
    Виявляє стійкі напрямлені відхилення сенсорів від початкового baseline двигуна.

    Перші baseline_cycles спостережень кожного двигуна калібрують baseline і не мають
    числових ознак детектора. fit() використовує лише тренувальні двигуни.
    """

    def __init__(
        self,
        baseline_cycles=20,
        window=10,
        persistence=5,
        threshold_quantile=0.99,
        min_scale=1e-6,
    ):
        if baseline_cycles < 1:
            raise ValueError("baseline_cycles must be >= 1")

        if window < 1:
            raise ValueError("window must be >= 1")

        if persistence < 1:
            raise ValueError("persistence must be >= 1")

        if not 0 < threshold_quantile < 1:
            raise ValueError("threshold_quantile must be between 0 and 1")

        self.baseline_cycles = baseline_cycles
        self.window = window
        self.persistence = persistence
        self.threshold_quantile = threshold_quantile
        self.min_scale = min_scale

        # +1: зростання сенсора додає до score; -1: спадання додає до score
        self.directions = {
            "sensor_4": 1,
            "sensor_7": -1,
            "sensor_11": 1,
            "sensor_12": -1,
            "sensor_15": 1,
            "sensor_20": -1,
            "sensor_21": -1,
        }

        self.sensors = list(self.directions)

        self.scale_ = None
        self.threshold_ = None
        self.is_fitted_ = False

    def _validate_columns(self, df):
        """Перевіряє наявність потрібних колонок."""
        required = {"unit_id", "cycle", *self.sensors}

        missing = required - set(df.columns)

        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

    def _calculate_scores(self, engine):
        """
        Відхилення сенсорів від baseline двигуна, сирий і згладжений score.
        Згладжування: ковзне середнє лише по поточній і минулих точках.
        """

        engine = engine.sort_values("cycle").copy()

        # Baseline двигуна: медіана перших baseline_cycles спостережень
        baseline = engine[self.sensors].iloc[:self.baseline_cycles].median()

        deviation = (
            engine[self.sensors]
            .sub(baseline, axis="columns")
            .div(self.scale_, axis="columns")
        )

        # Вирівнюємо відхилення з очікуваним напрямом деградації
        direction = pd.Series(self.directions)

        directional_deviation = (
            deviation
            .mul(direction, axis="columns")
            .clip(lower=0)
        )

        raw_score = directional_deviation.mean(axis=1)

        smooth_score = raw_score.rolling(window=self.window, min_periods=1).mean()

        return engine, deviation, raw_score, smooth_score

    def fit(self, train_df):
        """
        Оцінює глобальні масштаби сенсорів і поріг за калібраційними вікнами тренувальних двигунів.
        """

        self._validate_columns(train_df)

        calibration_residuals = []

        # Залишки відносно медіани перших baseline_cycles спостережень кожного двигуна
        for _, group in train_df.groupby("unit_id", sort=False):
            engine = group.sort_values("cycle")

            if len(engine) < self.baseline_cycles:
                continue

            calibration = engine.iloc[:self.baseline_cycles]
            baseline = calibration[self.sensors].median()

            residuals = calibration[self.sensors].sub(baseline, axis="columns")
            calibration_residuals.append(residuals)

        if not calibration_residuals:
            raise ValueError("No engine has enough observations for calibration")

        residuals_all = pd.concat(calibration_residuals, ignore_index=True)

        # Робастний масштаб (MAD * 1.4826) для кожного сенсора
        self.scale_ = (
            residuals_all
            .abs()
            .median()
            .mul(1.4826)
            .clip(lower=self.min_scale)
            .reindex(self.sensors)
        )

        calibration_scores = []

        for _, group in train_df.groupby("unit_id", sort=False):
            engine = group.sort_values("cycle")

            if len(engine) < self.baseline_cycles:
                continue

            _, _, _, smooth_score = self._calculate_scores(engine)
            calibration_scores.extend(smooth_score.iloc[:self.baseline_cycles].to_numpy())

        calibration_scores = np.asarray(calibration_scores, dtype=float)
        calibration_scores = calibration_scores[np.isfinite(calibration_scores)]

        if len(calibration_scores) == 0:
            raise ValueError("Could not calculate valid calibration scores")

        self.threshold_ = float(np.quantile(calibration_scores, self.threshold_quantile))

        self.n_calibration_scores_ = len(calibration_scores)
        self.is_fitted_ = True

        return self

    def transform(self, df):
        """
        Причинні ознаки детектора для кожного двигуна.
        Підтвердження деградації лише після persistence послідовних перевищень, без backfill.
        """

        if not self.is_fitted_:
            raise RuntimeError("Call fit(train_df) before transform(df)")

        self._validate_columns(df)

        results = []

        for _, group in df.groupby("unit_id", sort=False):
            engine, deviation, _, smooth_score = self._calculate_scores(group)

            n = len(engine)

            cycles = engine["cycle"].to_numpy(dtype=float)
            scores = smooth_score.to_numpy(dtype=float)
            deviations = deviation.to_numpy(dtype=float)

            degradation_detected = np.zeros(n, dtype=np.int8)

            # Числові ознаки недоступні під час калібрування
            degradation_score = np.full(n, np.nan)
            threshold_excess = np.full(n, np.nan)

            degradation_detection_cycle = np.full(n, np.nan)
            degradation_start_estimate = np.full(n, np.nan)

            cycles_since_detection = np.full(n, np.nan)
            cycles_since_degradation = np.full(n, np.nan)

            trend_since_degradation = np.full(n, np.nan)

            sensor_delta = {sensor: np.full(n, np.nan) for sensor in self.sensors}

            for i in range(self.baseline_cycles, n):
                degradation_score[i] = scores[i]
                threshold_excess[i] = max(0.0, scores[i] - self.threshold_)

            consecutive = 0
            detected = False

            detection_idx = None
            onset_idx = None

            for i in range(self.baseline_cycles, n):

                if not detected:
                    if scores[i] > self.threshold_:
                        consecutive += 1
                    else:
                        consecutive = 0

                    # Підтвердження лише після повної послідовності
                    if consecutive >= self.persistence:
                        detected = True
                        detection_idx = i
                        onset_idx = i - self.persistence + 1

                if detected:
                    degradation_detected[i] = 1

                    degradation_detection_cycle[i] = cycles[detection_idx]
                    degradation_start_estimate[i] = cycles[onset_idx]

                    cycles_since_detection[i] = cycles[i] - cycles[detection_idx]
                    cycles_since_degradation[i] = cycles[i] - cycles[onset_idx]

                    # Нахил згладженого score від оцінки початку до поточного циклу
                    history_cycles = cycles[onset_idx:i + 1]
                    history_scores = scores[onset_idx:i + 1]

                    if len(history_scores) >= 2 and np.ptp(history_cycles) > 0:
                        trend_since_degradation[i] = np.polyfit(history_cycles, history_scores, 1)[0]
                    else:
                        trend_since_degradation[i] = 0.0

                    delta = deviations[i] - deviations[onset_idx]

                    for j, sensor in enumerate(self.sensors):
                        sensor_delta[sensor][i] = delta[j]

            features = pd.DataFrame({"degradation_score": degradation_score}, index=engine.index)

            results.append(features)

        return pd.concat(results).reindex(df.index)

In [ ]:
def add_cto_v2_selected_features(
    df,
    sensors,
    short_window=5,
    long_window=15,
    depth=4,
    base_step=12,
    min_step=2,
    max_step=18,
    shrink=0.78,
    gamma=1.5,
    history_decay=0.75,
    eps=1e-8,
):
    """
    CTO v2, лише ознаки expected_trend і multiscale_disagreement для заданих сенсорів.
    Кожен двигун обробляється окремо; використовується лише минуле кожної точки.
    """

    df = df.copy()

    def linear_slope(values):
        """Нахил прямої МНК за індексом часу."""
        n = len(values)
        if n < 2:
            return 0.0

        t = np.arange(n, dtype=float)
        y = np.asarray(values, dtype=float)
        t_centered = t - t.mean()
        denominator = np.sum(t_centered ** 2)
        if denominator < eps:
            return 0.0

        return np.sum(t_centered * (y - y.mean())) / denominator

    def causal_state(values, idx):
        """Стан точки idx за її минулою історією (long_slope, disagreement, activity)."""
        short_hist = values[max(0, idx - short_window + 1):idx + 1]
        long_hist = values[max(0, idx - long_window + 1):idx + 1]

        long_slope = linear_slope(long_hist)
        disagreement = linear_slope(short_hist) - long_slope

        volatility = np.std(np.diff(long_hist)) if len(long_hist) >= 3 else 0.0

        local_scale = np.std(long_hist)
        if local_scale < eps:
            activity = 0.0
        else:
            activity = (abs(disagreement) + volatility) / (local_scale + eps)
        activity = min(activity, 10.0)

        return {"long_slope": long_slope, "disagreement": disagreement, "activity": activity}

    for sensor in sensors:
        expected_col = f"{sensor}_cto2_expected_trend"
        disagreement_col = f"{sensor}_cto2_multiscale_disagreement"

        df[expected_col] = np.nan
        df[disagreement_col] = np.nan

        for _, group in df.groupby("unit_id", sort=False):
            group = group.sort_values("cycle")

            values = group[sensor].to_numpy(dtype=float)
            row_indices = group.index.to_numpy()

            for current_idx in range(len(values)):

                if current_idx < max(short_window, min_step):
                    continue

                idx = current_idx
                current_base_step = float(base_step)

                expected_rates = []
                disagreements = []
                weights = []

                # A <- B <- C <- ...
                for level in range(depth):

                    state_current = causal_state(values, idx)

                    adaptive_step = current_base_step / (1.0 + gamma * state_current["activity"])
                    adaptive_step = int(np.round(adaptive_step))
                    adaptive_step = int(np.clip(adaptive_step, min_step, max_step))

                    previous_idx = idx - adaptive_step
                    if previous_idx < 0:
                        break

                    state_previous = causal_state(values, previous_idx)

                    expected_rates.append(state_previous["long_slope"])
                    disagreements.append(abs(state_previous["disagreement"]))
                    weights.append(history_decay ** level)

                    idx = previous_idx
                    current_base_step = max(min_step, current_base_step * shrink)

                # Потрібні щонайменше два історичні переходи
                if len(expected_rates) < 2:
                    continue

                expected_rates = np.asarray(expected_rates, dtype=float)
                disagreements = np.asarray(disagreements, dtype=float)
                weights = np.asarray(weights, dtype=float)
                weights = weights / (weights.sum() + eps)

                expected_trend = np.sum(weights * expected_rates)
                multiscale_disagreement = np.sum(weights * disagreements)

                row_idx = row_indices[current_idx]

                df.loc[row_idx, expected_col] = expected_trend
                df.loc[row_idx, disagreement_col] = multiscale_disagreement

    return df

In [ ]:
# Дані для ознак: без RUL, щоб уникнути випадкового використання цілі
train_raw = train_df.drop(columns=["RUL"])
val_raw = val_df.drop(columns=["RUL"])

# Wavelet: нові колонки, що додає функція
train_wavelet = add_wavelet_d2_std(train_raw)
val_wavelet = add_wavelet_d2_std(val_raw)
WAVELET_FEATURES = [col for col in train_wavelet.columns if col not in train_raw.columns]

# Degradation: детектор калібрується тільки на тренувальних двигунах
detector = DegradationDetector(baseline_cycles=20, window=10, persistence=5, threshold_quantile=0.99)
detector.fit(train_df)

train_deg = detector.transform(train_df)[["degradation_score"]]
val_deg = detector.transform(val_df)[["degradation_score"]]

# CTO v2: expected_trend і multiscale_disagreement для top-8 сенсорів
train_cto = add_cto_v2_selected_features(train_raw, sensors=TOP8_SENSORS)
val_cto = add_cto_v2_selected_features(val_raw, sensors=TOP8_SENSORS)

CTO_FEATURES = (
    [f"{s}_cto2_expected_trend" for s in TOP8_SENSORS]
    + [f"{s}_cto2_multiscale_disagreement" for s in TOP8_SENSORS]
)

# Блоки ознак; порядок колонок у кожному наборі задається в EXPERIMENTS
BLOCKS_TRAIN = {
    "base": train_raw[BASELINE_FEATURES],
    "wav": train_wavelet[WAVELET_FEATURES],
    "deg": train_deg,
    "cto": train_cto[CTO_FEATURES],
}

BLOCKS_VAL = {
    "base": val_raw[BASELINE_FEATURES],
    "wav": val_wavelet[WAVELET_FEATURES],
    "deg": val_deg,
    "cto": val_cto[CTO_FEATURES],
}

print("Baseline:", len(BASELINE_FEATURES), "| wavelet:", len(WAVELET_FEATURES),
      "| degradation:", 1, "| CTO:", len(CTO_FEATURES))

## 3. Експерименти

In [ ]:
XGB_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.012074507989018332,
    "max_depth": 4,
    "min_child_weight": 3.2298041995296414,
    "subsample": 0.7900204897773602,
    "colsample_bytree": 0.9223090698240903,
    "reg_alpha": 0.7459635164536705,
    "reg_lambda": 4.416722489003329,
    "gamma": 0.2695542352154229,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
}

# Набори з блоків у тому самому порядку колонок, що й в оригінальних експериментах
CONFIGS = {
    "Baseline": ["base"],
    "Baseline + Degradation": ["base", "deg"],
    "Baseline + Wavelet": ["base", "wav"],
    "Baseline + CTO": ["base", "cto"],
    "Baseline + Degradation + Wavelet": ["base", "deg", "wav"],
    "Baseline + Degradation + CTO": ["base", "deg", "cto"],
    "Baseline + Wavelet + CTO": ["base", "wav", "cto"],
    "All Features": ["base", "wav", "deg", "cto"],
}


def run_configs(configs, blocks_train, blocks_val, y_tr, y_v):
    """Навчає XGBoost на кожному наборі блоків і повертає MAE, RMSE, R2 на validation."""
    rows = []
    for name, keys in configs.items():
        X_tr = pd.concat([blocks_train[k] for k in keys], axis=1)
        X_v = pd.concat([blocks_val[k] for k in keys], axis=1)

        assert X_tr.columns.is_unique, f"{name}: дублікати колонок у train"
        assert X_v.columns.equals(X_tr.columns), f"{name}: колонки train і validation відрізняються"

        model = XGBRegressor(**XGB_PARAMS)
        model.fit(X_tr, y_tr)

        pred = np.maximum(model.predict(X_v), 0)

        rows.append({
            "Experiment": name,
            "Features": X_tr.shape[1],
            "MAE": mean_absolute_error(y_v, pred),
            "RMSE": np.sqrt(mean_squared_error(y_v, pred)),
            "R2": r2_score(y_v, pred),
        })

    return pd.DataFrame(rows)

## 4. Результати

In [ ]:
results = run_configs(CONFIGS, BLOCKS_TRAIN, BLOCKS_VAL, y_train, y_val)

baseline_mae = results.loc[results["Experiment"] == "Baseline", "MAE"].iloc[0]
results["MAE improvement"] = baseline_mae - results["MAE"]
results["MAE improvement %"] = results["MAE improvement"] / baseline_mae * 100

results = results.sort_values("MAE").reset_index(drop=True)
display(results.round(4))

## Висновки

Значення з оригінальних виводів ноутбука (validation: MAE / RMSE / R²):

- **Baseline** (25): 23.0717 / 29.8586 / 0.7932.
- **+ Degradation** (+1, `degradation_score`): 18.9638 / 25.4928 / 0.8492. Найбільший внесок: MAE −4.1079.
- **+ Wavelet** (+7): 22.9140 / 29.6756 / 0.7957. Слабкий ефект: MAE −0.1577.
- **+ CTO** (+16, expected_trend і multiscale_disagreement): 22.3040 / 29.1135 / 0.8034. MAE −0.7676.
- **+ Degradation + Wavelet** (33): 18.9606 / 25.5362 / 0.8487. Найнижчий MAE у таблиці; wavelet поверх degradation дає лише −0.0032.
- **+ Degradation + CTO** (42): 18.9993 / 25.6007 / 0.8479. CTO поверх degradation трохи погіршує MAE (+0.0355).
- **+ Wavelet + CTO** (48): 22.2962 / 29.0819 / 0.8038.
- **All features** (49): 18.9801 / 25.5876 / 0.8481. Гірше за Degradation + Wavelet (18.9606).

Основний ефект дає `degradation_score`. Wavelet і CTO поверх нього не дають стабільного покращення: різниці в межах 0.04 MAE.